In [2]:
import zipfile
import os

zip_path = "../data/SWPK.zip"

with zipfile.ZipFile(zip_path, "r") as zip_file:
    print("Files inside the ZIP:")
    for file in zip_file.namelist():
        print(file)

Files inside the ZIP:
datadump_s5-000.csv
datadump_s5-001.csv
datadump_s5-002.csv
datadump_s5-003.csv
datadump_s5-004.csv
datadump_s5-005.csv
datadump_s5-006.csv
datadump_s5-007.csv
datadump_s5-008.csv
datadump_s5-009.csv
datadump_s5-010.csv
datadump_s5-011.csv
datadump_s5-012.csv
datadump_s5-013.csv
datadump_s5-014.csv
datadump_s5-015.csv
datadump_s5-016.csv
datadump_s5-017.csv
datadump_s5-018.csv
datadump_s5-019.csv
datadump_s5-020.csv
datadump_s5-021.csv


In [3]:
import zipfile
import pandas as pd

zip_path = "../data/SWPK.zip"

with zipfile.ZipFile(zip_path, "r") as zip_file:
    with zip_file.open("datadump_s5-000.csv") as csv_file:
        df = pd.read_csv(csv_file, nrows=1000)

print("Rows:", len(df))
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Rows: 1000

Columns:
['dateid', 'platform', 'gamemode', 'mapname', 'matchid', 'roundnumber', 'objectivelocation', 'winrole', 'endroundreason', 'roundduration', 'clearancelevel', 'skillrank', 'role', 'team', 'haswon', 'operator', 'nbkills', 'isdead', 'primaryweapon', 'primaryweapontype', 'primarysight', 'primarygrip', 'primaryunderbarrel', 'primarybarrel', 'secondaryweapon', 'secondaryweapontype', 'secondarysight', 'secondarygrip', 'secondaryunderbarrel', 'secondarybarrel', 'secondarygadget']

First 5 rows:


,dateid,platform,gamemode,mapname,matchid,roundnumber,objectivelocation,winrole,endroundreason,roundduration,...,primarygrip,primaryunderbarrel,primarybarrel,secondaryweapon,secondaryweapontype,secondarysight,secondarygrip,secondaryunderbarrel,secondarybarrel,secondarygadget
0,20170212,PC,HOSTAGE,CLUB_HOUSE,1522380841,1,STRIP_CLUB,Defender,AttackersKilledHostage,124,...,Vertical,NaN,Compensator,5.7_USG,Pistols,NaN,NaN,NaN,NaN,IMPACT_GRENADE
1,20170212,PC,HOSTAGE,CLUB_HOUSE,1522380841,4,CHURCH,Defender,AttackersEliminated,217,...,Vertical,Laser,Suppressor,P12,Pistols,NaN,NaN,Laser,Suppressor,DEPLOYABLE_SHIELD
2,20170212,PC,HOSTAGE,CLUB_HOUSE,1522380841,3,CHURCH,Defender,AttackersEliminated,160,...,NaN,NaN,NaN,MK1_9mm,Pistols,NaN,NaN,NaN,NaN,DEPLOYABLE_SHIELD
3,20170212,PC,HOSTAGE,CLUB_HOUSE,1522380841,4,CHURCH,Defender,AttackersEliminated,217,...,NaN,NaN,MuzzleBrake,PRB92,Pistols,NaN,NaN,NaN,NaN,IMPACT_GRENADE
4,20170212,PC,HOSTAGE,CLUB_HOUSE,1522380841,6,BEDROOM,Attacker,DefendersEliminated,143,...,Vertical,Laser,Suppressor,P12,Pistols,NaN,NaN,Laser,Suppressor,DEPLOYABLE_SHIELD


In [4]:
# Look at one specific match and round
sample = df[
    (df["matchid"] == 1522380841) &
    (df["roundnumber"] == 4)
]

print(sample[["matchid", "roundnumber", "role", "team", "operator", "winrole", "haswon"]])
print("\nNumber of players:", len(sample))

       matchid  roundnumber      role  team             operator   winrole  \
1   1522380841            4  Defender     0           GSG9-JAGER  Defender   
3   1522380841            4  Defender     0         BOPE-CAVEIRA  Defender   
15  1522380841            4  Attacker     1              GSG9-IQ  Defender   
17  1522380841            4  Attacker     1  NAVYSEAL-BLACKBEARD  Defender   
20  1522380841            4  Defender     0    SPETSNAZ-TACHANKA  Defender   
21  1522380841            4  Defender     0          GSG9-BANDIT  Defender   
27  1522380841            4  Attacker     1           SAT-HIBANA  Defender   
34  1522380841            4  Attacker     1        G.E.O.-JACKAL  Defender   
51  1522380841            4  Attacker     1             SWAT-ASH  Defender   

    haswon  
1        1  
3        1  
15       0  
17       0  
20       1  
21       1  
27       0  
34       0  
51       0  

Number of players: 9


In [5]:
# Count how many attackers and defenders each round has

round_counts = (
    df.groupby(["matchid", "roundnumber", "role"])
      .size()
      .unstack(fill_value=0)
)

print(round_counts.head(20))

role                    Attacker  Defender
matchid    roundnumber                    
1522380841 1                   5         5
           2                   5         4
           3                   3         5
           4                   5         4
           5                   4         5
           6                   5         4
1522488121 1                   5         5
           2                   5         5
           3                   5         5
           4                   5         5
1522514641 1                   5         5
           2                   4         5
           3                   5         4
           4                   4         5
1522741281 1                   4         5
           2                   5         4
           3                   4         5
           4                   5         4
           5                   4         5
1522760761 1                   5         5


In [6]:
# Look at the relationship between winrole and haswon
print(
    df[["role", "winrole", "haswon"]]
    .drop_duplicates()
    .sort_values(["role", "winrole", "haswon"])
)

       role   winrole  haswon
6  Attacker  Attacker       1
8  Attacker  Defender       0
4  Defender  Attacker       0
0  Defender  Defender       1


In [7]:
# Check whether each round has one consistent winner
winner_check = (
    df.groupby(["matchid", "roundnumber"])["winrole"]
      .nunique()
)

print("Rounds with exactly one winner:", (winner_check == 1).sum())
print("Rounds with multiple winners:", (winner_check > 1).sum())

Rounds with exactly one winner: 108
Rounds with multiple winners: 0


In [8]:
# Count the number of attackers and defenders in each round
round_counts = (
    df.groupby(["matchid", "roundnumber", "role"])
      .size()
      .unstack(fill_value=0)
)

# Keep only complete 5v5 rounds
complete_rounds = round_counts[
    (round_counts["Attacker"] == 5) &
    (round_counts["Defender"] == 5)
]

print("Complete 5v5 rounds:", len(complete_rounds))

Complete 5v5 rounds: 65


In [9]:
# Get the IDs of all complete 5v5 rounds
complete_round_ids = complete_rounds.index

# Keep only rows belonging to those rounds
clean_df = df.set_index(["matchid", "roundnumber"]).loc[
    complete_round_ids
].reset_index()

print("Rows in cleaned dataset:", len(clean_df))
print(clean_df.head())

Rows in cleaned dataset: 650
      matchid  roundnumber    dateid platform gamemode     mapname  \
0  1522380841            1  20170212       PC  HOSTAGE  CLUB_HOUSE   
1  1522380841            1  20170212       PC  HOSTAGE  CLUB_HOUSE   
2  1522380841            1  20170212       PC  HOSTAGE  CLUB_HOUSE   
3  1522380841            1  20170212       PC  HOSTAGE  CLUB_HOUSE   
4  1522380841            1  20170212       PC  HOSTAGE  CLUB_HOUSE   

  objectivelocation   winrole          endroundreason  roundduration  ...  \
0        STRIP_CLUB  Defender  AttackersKilledHostage            124  ...   
1        STRIP_CLUB  Defender  AttackersKilledHostage            124  ...   
2        STRIP_CLUB  Defender  AttackersKilledHostage            124  ...   
3        STRIP_CLUB  Defender  AttackersKilledHostage            124  ...   
4        STRIP_CLUB  Defender  AttackersKilledHostage            124  ...   

   primarygrip primaryunderbarrel primarybarrel  secondaryweapon  \
0     Vertical     

In [10]:
# Pick one complete round
example_round = clean_df[
    (clean_df["matchid"] == 1522488121) &
    (clean_df["roundnumber"] == 1)
]

# Display only the columns we care about right now
print(
    example_round[
        [
            "matchid",
            "roundnumber",
            "mapname",
            "objectivelocation",
            "role",
            "operator",
            "winrole"
        ]
    ].to_string(index=False)
)

   matchid  roundnumber mapname             objectivelocation     role          operator  winrole
1522488121            1   PLANE MEETING_ROOM-EXECUTIVE_OFFICE Attacker     SWAT-THERMITE Defender
1522488121            1   PLANE MEETING_ROOM-EXECUTIVE_OFFICE Defender        SWAT-PULSE Defender
1522488121            1   PLANE MEETING_ROOM-EXECUTIVE_OFFICE Attacker     G.E.O.-JACKAL Defender
1522488121            1   PLANE MEETING_ROOM-EXECUTIVE_OFFICE Attacker     SPETSNAZ-GLAZ Defender
1522488121            1   PLANE MEETING_ROOM-EXECUTIVE_OFFICE Attacker        SAS-SLEDGE Defender
1522488121            1   PLANE MEETING_ROOM-EXECUTIVE_OFFICE Attacker     GIGN-MONTAGNE Defender
1522488121            1   PLANE MEETING_ROOM-EXECUTIVE_OFFICE Defender        JTF2-FROST Defender
1522488121            1   PLANE MEETING_ROOM-EXECUTIVE_OFFICE Defender        GSG9-JAGER Defender
1522488121            1   PLANE MEETING_ROOM-EXECUTIVE_OFFICE Defender NAVYSEAL-VALKYRIE Defender
1522488121          

In [11]:
# Get the attackers
attackers = example_round[
    example_round["role"] == "Attacker"
]["operator"].tolist()

# Get the defenders
defenders = example_round[
    example_round["role"] == "Defender"
]["operator"].tolist()

# Get the winner
winner = example_round["winrole"].iloc[0]

print("Attackers:")
print(attackers)

print("\nDefenders:")
print(defenders)

print("\nWinner:")
print(winner)

Attackers:
['SWAT-THERMITE', 'G.E.O.-JACKAL', 'SPETSNAZ-GLAZ', 'SAS-SLEDGE', 'GIGN-MONTAGNE']

Defenders:
['SWAT-PULSE', 'JTF2-FROST', 'GSG9-JAGER', 'NAVYSEAL-VALKYRIE', 'G.E.O.-MIRA']

Winner:
Defender
